In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV
from sklearn.feature_selection import SelectFromModel
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

# 1. Import hàm đánh giá từ file evaluate.py của TV3
from src.evaluate import evaluate_model

# 2. Đọc DỮ LIỆU THẬT từ TV2
print("Đang nạp dữ liệu...")
X_train = pd.read_csv('data/processed/X_train.csv')
y_train = pd.read_csv('data/processed/y_train.csv')
X_test = pd.read_csv('data/processed/X_test.csv')
y_test = pd.read_csv('data/processed/y_test.csv')

# Khởi tạo mô hình theo đúng siêu tham số cố định
models = {
    'SVR': SVR(),
    'RandomForest': RandomForestRegressor(random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(random_state=42, n_jobs=-1)
}

def get_embedded_lasso(model):
    selector = SelectFromModel(LassoCV(cv=5, random_state=42, n_jobs=-1))
    return Pipeline([('feature_selection', selector), ('model', model)])

def get_embedded_rf(model):
    rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    selector = SelectFromModel(rf, threshold=-np.inf, max_features=50) # Chọn top 50
    return Pipeline([('feature_selection', selector), ('model', model)])

# 3. CHẠY ĐÁNH GIÁ (BASELINE + EMBEDDED)
results = []

print("Đang chạy Baseline...")
for name, model in models.items():
    res = evaluate_model('baseline', name, model, X_train, y_train, X_test, y_test)
    results.append(res)

print("Đang chạy Embedded Lasso...")
for name, model in models.items():
    pipe_lasso = get_embedded_lasso(model)
    res = evaluate_model('embedded_lasso', name, pipe_lasso, X_train, y_train, X_test, y_test)
    results.append(res)

print("Đang chạy Embedded Random Forest...")
for name, model in models.items():
    pipe_rf = get_embedded_rf(model)
    res = evaluate_model('embedded_rf', name, pipe_rf, X_train, y_train, X_test, y_test)
    results.append(res)

# 4. LƯU KẾT QUẢ VÀ XUẤT HÌNH ẢNH
os.makedirs('data/processed', exist_ok=True)
os.makedirs('figures', exist_ok=True)

df_all = pd.DataFrame(results)
# Lọc riêng Embedded để lưu file theo phân công TV5
df_embedded = df_all[df_all['technique'].str.contains('embedded')]
df_embedded.to_csv('data/processed/fs_results_embedded.csv', index=False)
print("-> Đã lưu fs_results_embedded.csv")

# Hình 1: RMSE by technique
plt.figure(figsize=(10, 5))
sns.barplot(data=df_all, x='technique', y='cv_log_rmse', hue='model')
plt.title('CV Log RMSE Comparison')
plt.savefig('figures/rmse_by_technique.png', dpi=150)
plt.close()

# Hình 2: Delta Heatmap (% thay đổi so với Baseline)
pivot_all = df_all.pivot(index='technique', columns='model', values='cv_log_rmse')
baseline_series = pivot_all.loc['baseline']
delta_heatmap = (pivot_all - baseline_series) / baseline_series * 100

plt.figure(figsize=(8, 5))
sns.heatmap(delta_heatmap, annot=True, cmap='RdYlGn_r', center=0, fmt='.2f')
plt.title('Percentage Change in RMSE vs Baseline (%)')
plt.savefig('figures/delta_heatmap.png', dpi=150)
plt.close()

# Hình 3: Time comparison
plt.figure(figsize=(10, 5))
sns.barplot(data=df_all, x='technique', y='train_seconds', hue='model')
plt.title('Training Time Comparison (seconds)')
plt.savefig('figures/time_comparison.png', dpi=150)
plt.close()

# Hình 4: Feature Agreement (Lấy danh sách feature từ Pipeline Lasso và RF của mô hình cuối)
lasso_sel = pipe_lasso.named_steps['feature_selection'].get_support()
rf_sel = pipe_rf.named_steps['feature_selection'].get_support()

features = X_train.columns
lasso_features = features[lasso_sel].tolist()
rf_features = features[rf_sel].tolist()

from collections import Counter
feature_counts = Counter(lasso_features + rf_features)
df_features = pd.DataFrame(feature_counts.items(), columns=['Feature', 'Count']).sort_values('Count', ascending=False)
df_features.to_csv('data/processed/selected_features.csv', index=False)

plt.figure(figsize=(10, 6))
# Lấy top 15 đặc trưng được đồng thuận cao nhất
sns.barplot(data=df_features.head(15), x='Count', y='Feature', palette='viridis')
plt.title('Top Features Agreement Across Embedded Techniques')
plt.savefig('figures/feature_agreement.png', dpi=150)
plt.close()

print("HOÀN TẤT 100% CÔNG VIỆC CỦA TV5! Hãy kiểm tra thư mục data và figures.")